# About
This colab is an introduction to a RAG based ReAct agent that looks at a Coastal Karnataka (Mangalore/Udupi) recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0


In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [3]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai google-genai --quiet
print("Done.")

Done.


In [4]:
# @title Create a database of recipes.
RECIPES = {
    "neer_dosa": (
        "1 cup raw rice (sona masuri or any raw rice). "
        "1/2 cup freshly grated coconut. "
        "1 tsp salt. "
        "1.5 to 2 cups water as needed. "
        "1 tsp oil for greasing the pan. "
        "Soak the raw rice in water for 4 hours then drain. "
        "Grind the rice with the grated coconut and water into a very smooth and thin batter. The batter should be watery like buttermilk. Add salt and mix. "
        "No fermentation is needed. The batter can be used immediately. "
        "Heat a non-stick tawa on medium heat and grease it lightly with oil. "
        "Pour a ladle of batter from a height and swirl the pan to spread it into a thin lacy circle. Do not flip the dosa. "
        "Cover and cook for 1-2 minutes until the edges lift and the dosa turns soft and cooked. "
        "Fold into quarters and serve hot with coconut chutney or chicken curry."
    ),
    "chicken_ghee_roast": (
        "500 g chicken pieces. "
        "1/2 cup thick curd (yogurt). "
        "1 tsp turmeric powder. "
        "1 tsp salt. "
        "10-12 dried Byadgi red chillies. "
        "1 tbsp coriander seeds. "
        "1 tsp cumin seeds. "
        "1 tsp black peppercorns. "
        "1/2 tsp fenugreek seeds. "
        "6 garlic cloves. "
        "1 tbsp tamarind pulp. "
        "4 tbsp ghee. "
        "10 curry leaves. "
        "1 tbsp jaggery. "
        "Soak the dried red chillies in warm water for 20 minutes. Dry roast the coriander seeds cumin peppercorns and fenugreek for 2 minutes. "
        "Grind the chillies roasted spices garlic and tamarind into a smooth red paste. "
        "Mix the chicken with curd turmeric salt and half of the masala paste. Marinate for 1 hour. "
        "Heat 3 tbsp ghee in a heavy pan and add curry leaves. Add the marinated chicken and cook on medium heat for 20-25 minutes stirring often until the chicken is cooked and the masala turns dry. "
        "Add the remaining masala paste jaggery and 1 more tbsp of ghee. Roast for 5 minutes until the masala coats the chicken and turns dark red. "
        "Serve hot with neer dosa or rice."
    ),
    "mangalorean_fish_curry": (
        "500 g fish pieces such as kingfish or seer fish. "
        "1 cup freshly grated coconut. "
        "6 dried Byadgi red chillies. "
        "1 tbsp coriander seeds. "
        "1/2 tsp cumin seeds. "
        "1/2 tsp fenugreek seeds. "
        "1/2 tsp turmeric powder. "
        "1 small lemon-sized ball of tamarind. "
        "1 onion finely chopped. "
        "3 tbsp coconut oil. "
        "1 tsp mustard seeds. "
        "10 curry leaves. "
        "Salt to taste. "
        "Dry roast the red chillies coriander seeds cumin and fenugreek for 2 minutes. "
        "Grind the roasted spices with coconut turmeric and tamarind into a smooth paste using a little water. "
        "Heat coconut oil in a pot. Add mustard seeds and curry leaves. When the seeds splutter add the onion and cook for 4 minutes until soft. "
        "Add the ground masala paste 2 cups of water and salt. Simmer on medium heat for 10 minutes. "
        "Add the fish pieces and cook on low heat for 8-10 minutes until the fish is cooked. Do not stir hard or the fish will break. "
        "Serve hot with steamed rice."
    ),
    "kori_rotti": (
        "12 pieces of crisp kori rotti (dry rice wafers). "
        "500 g chicken pieces. "
        "1 cup freshly grated coconut. "
        "8 dried red chillies. "
        "1 tbsp coriander seeds. "
        "1 tsp cumin seeds. "
        "1/2 tsp fenugreek seeds. "
        "1/2 tsp turmeric powder. "
        "1 tbsp tamarind pulp. "
        "2 onions finely chopped. "
        "1 tomato chopped. "
        "5 garlic cloves. "
        "2 tbsp coconut oil. "
        "Salt to taste. "
        "Dry roast the red chillies coriander seeds cumin and fenugreek for 2 minutes. Grind them with coconut turmeric and tamarind into a smooth paste. "
        "Heat coconut oil in a pot and cook the onions and garlic for 5 minutes until golden. Add the tomato and cook until soft. "
        "Add the chicken and cook for 5 minutes. Add the ground masala paste 1.5 cups of water and salt. "
        "Cover and simmer for 25 minutes until the chicken is cooked and the gravy is thick. "
        "Tear the kori rotti into bite-sized pieces and place them in a serving bowl. "
        "Pour the hot chicken gravy over the rotti only at the time of serving so that it stays crisp. Serve immediately."
    ),
    "masala_dosa": (
        "3 cups dosa rice (or parboiled rice). "
        "1 cup urad dal (split black gram). "
        "1/2 tsp fenugreek seeds (methi). "
        "1 tsp salt. "
        "3 tbsp ghee or oil for cooking. "
        "4 medium potatoes boiled and mashed. "
        "1 large onion thinly sliced. "
        "1 tsp mustard seeds. "
        "1 tsp urad dal for tempering. "
        "10 curry leaves. "
        "2 green chillies chopped. "
        "1/2 tsp turmeric powder. "
        "Soak the rice and the urad dal with fenugreek seeds separately in water for 5 hours. "
        "Grind the urad dal into a fluffy batter and the rice into a slightly coarse batter. Mix both batters with salt. "
        "Cover and let the batter ferment in a warm place for 8-12 hours or overnight until it rises and smells slightly sour. "
        "For the potato filling heat 1 tbsp oil and add mustard seeds. When they splutter add urad dal curry leaves and green chillies. "
        "Add the onion and cook for 5 minutes until soft. Add turmeric and salt then mix in the mashed potatoes and cook for 3 minutes. "
        "Heat a flat tawa on medium-high heat. Pour a ladle of batter in the centre and spread it in a thin circle. "
        "Drizzle ghee around the edges and cook for 2-3 minutes until the dosa is golden brown and crisp. "
        "Place a spoonful of potato filling in the middle fold the dosa and serve hot with coconut chutney and sambar."
    )
}

In [5]:
# @title Define a tool
def search_recipes(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in RECIPES.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Quick test
print(search_recipes("coconut oil"))


[Source: neer_dosa]
1 cup raw rice (sona masuri or any raw rice). 1/2 cup freshly grated coconut. 1 tsp salt. 1.5 to 2 cups water as needed. 1 tsp oil for greasing the pan. Soak the raw rice in water for 4 hours then drain. Grind the rice with the grated coconut and water into a very smooth and thin batter. The batter should be watery like buttermilk. Add salt and mix. No fermentation is needed. The batter can be used immediately. Heat a non-stick tawa on medium heat and grease it lightly with oil. Pour a ladle of batter from a height and swirl the pan to spread it into a thin lacy circle. Do not flip the dosa. Cover and cook for 1-2 minutes until the edges lift and the dosa turns soft and cooked. Fold into quarters and serve hot with coconut chutney or chicken curry.

[Source: masala_dosa]
3 cups dosa rice (or parboiled rice). 1 cup urad dal (split black gram). 1/2 tsp fenugreek seeds (methi). 1 tsp salt. 3 tbsp ghee or oil for cooking. 4 medium potatoes boiled and mashed. 1 large oni

In [6]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_recipes],
    system_instruction=(
        '''
        You are a chef training at a coastal Karnataka restaurant in Mangaluru.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_recipes tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


/tmp/ipykernel_2966/4088246420.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_recipes",
        description="Search the cookbook to find most the relevant recipe to cook.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_recipes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What can I cook with coconut and rice?")

Question: What can I cook with coconut and rice?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'coconut and rice'})
  Result preview: [Source: neer_dosa]
1 cup raw rice (sona masuri or any raw rice). 1/2 cup freshly grated coconut. 1 ...

Final answer:
Coconut and rice make a fantastic combination in many traditional dishes! Here is one great recipe you can make with them:

### **Neer Dosa**
A wonderfully soft, delicate, and lacy crepe from coastal Karnataka made by grinding together rice and fresh coconut. It requires no fermentation and comes together easily.

#### **Ingredients:**
* 1 cup raw rice (Sona Masuri or any standard raw rice)
* ½ cup freshly grated coconut
* 1 tsp salt
* 1.5 to 2 cups water (as needed)
* 1 tsp oil (for greasing the pan)

#### **Instructions:**
1. **Soak the Rice:** Soak the raw rice in water for about 4 hours, then drain.
2. **Grind:** In a blender, grind the soaked rice along with the grated coconut and so

In [8]:
questions = [
    "Which dish on our menu uses fish and how long is the fish cooked?",
    "How long should the masala dosa batter ferment?",
    "Does the neer dosa batter need fermentation?",
    "How much ghee does the chicken ghee roast need?",
    "How do we keep the kori rotti crisp when serving?",
    # This question is not answered by our database.
    "How many customers order our fish curry everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most famous street food in Mumbai?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Which dish on our menu uses fish and how long is the fish cooked?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'fish'})
  Result preview: [Source: mangalorean_fish_curry]
500 g fish pieces such as kingfish or seer fish. 1 cup freshly grat...

Final answer:
The **Mangalorean Fish Curry** uses fish (such as kingfish or seer fish). 

The fish pieces are added to the curry and cooked on low heat for **8 to 10 minutes** until done.

Question: How long should the masala dosa batter ferment?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'masala dosa batter ferment'})
  Result preview: [Source: neer_dosa]
1 cup raw rice (sona masuri or any raw rice). 1/2 cup freshly grated coconut. 1 ...

Final answer:
For a traditional masala dosa, the batter should ferment in a warm place for **8 to 12 hours** (or overnight) until it rises, becomes light and airy, and develops a slightly pleasant, 

In [9]:
# This cell will check if you have added at least 4 questions.
assert(len(questions) >= 4)
# This will check if you replaced the placeholder question.
assert("Add a question here." not in questions)

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.